## Fast and efficient use of datacubes - parallell processing

This tutorial demonstrates how to optimize your data processing of the Copernicus Sentinel-2 datacubes in netCDF-CF provided through [satellittdata.no](https://www.satellittdata.no/) - the data portal for Norway’s national ground segment for satellite data (NBS). 

In particular, we look into parallell processing combining `dask` and `xarray`.


## Importing libraries

Note that the tuturial requires some standard libraries which can be installed by `pip install ...`

In [1]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import time

from dask.distributed import Client, LocalCluster
client = None
cluster = None

# For georeferencing
import cartopy.crs as ccrs

In [2]:
%env HDF5_USE_FILE_LOCKING=FALSE
import dask
import netCDF4
dask.__version__,xr.__version__,netCDF4.__version__





env: HDF5_USE_FILE_LOCKING=FALSE


('2025.11.0', '2025.6.1', '1.6.3')

In [3]:
import os
print(os.getenv('HDF5_USE_FILE_LOCKING'))

FALSE


In [4]:
# List of example data cubes
available_dcs = ['https://nbstds.met.no/thredds/dodsC/NetCDF-ondemand-products/datacubes/L2A/T32VNM/datacubes/S2_L2A_T32VNM_2025.ncml',
                 'https://nbstds.met.no/thredds/dodsC/NetCDF-ondemand-products/datacubes/L2A/T32VNM/datacubes/S2_L2A_T32VNM_2024.ncml',
                 'https://nbstds.met.no/thredds/dodsC/NetCDF-ondemand-products/datacubes/L2A/T32VNM/datacubes/S2_L2A_T32VNM_2019.ncml'
                ]


## Initialize local dask cluster


In [5]:
# Close the cluster and client if open
if client is not None:
    client.close()
if cluster is not None:
    cluster.close()
# Create a local Dask cluster
cluster = LocalCluster(memory_limit="2GiB")
# Connect a client to the cluster
client = Client(cluster)
client

/home/trygveh/miniconda3/envs/nbs/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41099 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:41099/status,
Dashboard: http://127.0.0.1:41099/status,Workers: 4
Total threads: 4,Total memory: 8.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43691,Workers: 0
Dashboard: http://127.0.0.1:41099/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:39935,Total threads: 1
Dashboard: http://127.0.0.1:43627/status,Memory: 2.00 GiB
Nanny: tcp://127.0.0.1:39717,


2025-12-03 15:45:32,091 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:35299 (pid=21176) exceeded 95% memory budget. Restarting...
2025-12-03 15:45:32,170 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:35299' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('getitem-99b5ac344e3806aa799cc234786f70a9', 3, 1, 1), ('getitem-99b5ac344e3806aa799cc234786f70a9', 4, 1, 1), ('getitem-99b5ac344e3806aa799cc234786f70a9', 2, 1, 0), ('getitem-99b5ac344e3806aa799cc234786f70a9', 1, 1, 0), ('getitem-99b5ac344e3806aa799cc234786f70a9', 0, 1, 1), ('getitem-99b5ac344e3806aa799cc234786f70a9', 2, 0, 1), ('getitem-99b5ac344e3806aa799cc234786f70a9', 3, 1, 0), ('getitem-99b5ac344e3806aa799cc234786f70a9', 1, 0, 1), ('getitem-99b5ac344e3806aa799cc234786f70a9', 4, 1, 0), ('getitem-99b5ac344e3806aa799cc234786f70a9', 4, 0, 1), ('getitem-99b5ac344e3806aa799cc234786f70a9', 2, 0, 0), ('getitem-99b5ac344e3806aa799cc234786f70a9', 0, 0, 1), ('ge

## Xarray without using dask

Dask seems not to be loaded if chunks is not specified as a keword argument


In [6]:
%%time
start_time = time.time()
ds = xr.open_dataset(available_dcs[2], engine='netcdf4')
end_time = time.time()
duration = end_time - start_time
print(f"The process took {duration:.6f} seconds.")

The process took 4.078041 seconds.
CPU times: user 761 ms, sys: 147 ms, total: 907 ms
Wall time: 4.08 s


In [7]:
%%time
# Read band information
b4 = ds['B4'].isel(time=slice(1,100),x=slice(0,2000), y=slice(0,2000))

CPU times: user 576 μs, sys: 90 μs, total: 666 μs
Wall time: 677 μs


In [8]:
%%time
# Read pixel values
inter_val = b4.values

CPU times: user 1min 16s, sys: 37.9 s, total: 1min 54s
Wall time: 6min 16s


In [9]:
del b4
ds.close()

## Xarray with dask

In [10]:
%%time
start_time = time.time()
chunks = {'time': 10, 'x': 2745, 'y': 2745}
chunks ='auto'
ds = xr.open_dataset(available_dcs[2], engine='netcdf4', chunks=chunks)

end_time = time.time()
duration = end_time - start_time
print(f"The process took {duration:.6f} seconds.")

The process took 7.678832 seconds.
CPU times: user 1.94 s, sys: 450 ms, total: 2.39 s
Wall time: 7.68 s


In [11]:
%%time
b4 = ds['B4'].isel(time=slice(1,100),x=slice(0,2000), y=slice(0,2000))
b4.chunk({'time': 10, 'x': 2745, 'y': 2745})

CPU times: user 8.41 ms, sys: 147 μs, total: 8.56 ms
Wall time: 7.87 ms


<xarray.DataArray 'B4' (time: 99, y: 2000, x: 2000)> Size: 2GB
dask.array<rechunk-merge, shape=(99, 2000, 2000), dtype=float32, chunksize=(10, 2000, 2000), chunktype=numpy.ndarray>
Coordinates:
  * x        (x) int32 8kB 499980 499990 500000 500010 ... 519950 519960 519970
  * y        (y) int32 8kB 6700020 6700010 6700000 ... 6680050 6680040 6680030
  * time     (time) datetime64[ns] 792B 2019-07-02T10:40:31 ... 2019-11-17T10...
Attributes:
    units:                  1
    grid_mapping:           UTM_projection
    standard_name:          surface_bidirectional_reflectance
    long_name:              Reflectance in band B4
    bandwidth:              30
    bandwidth_unit:         nm
    wavelength:             665
    wavelength_unit:        nm
    solar_irradiance:       1512.79
    solar_irradiance_unit:  W/m2/um
    _ChunkSizes:            [   1 2745 2745]

In [12]:
from dask.diagnostics import ProgressBar


In [13]:
%%time
with ProgressBar():
    inter_val = b4.values

2025-12-03 15:45:32,017 - distributed.worker.memory - WARNING - Worker is at 82% memory usage. Pausing worker.  Process memory: 1.65 GiB -- Worker memory limit: 2.00 GiB
2025-12-03 15:45:32,175 - distributed.nanny - WARNING - Restarting worker
2025-12-03 15:47:36,592 - distributed.worker.memory - WARNING - Worker is at 82% memory usage. Pausing worker.  Process memory: 1.65 GiB -- Worker memory limit: 2.00 GiB
2025-12-03 15:47:36,761 - distributed.nanny - WARNING - Restarting worker
2025-12-03 15:51:03,731 - distributed.worker.memory - WARNING - Worker is at 88% memory usage. Pausing worker.  Process memory: 1.78 GiB -- Worker memory limit: 2.00 GiB
2025-12-03 15:51:03,943 - distributed.nanny - WARNING - Restarting worker
2025-12-03 15:52:52,250 - distributed.worker.memory - WARNING - Worker is at 82% memory usage. Pausing worker.  Process memory: 1.65 GiB -- Worker memory limit: 2.00 GiB
2025-12-03 15:52:52,718 - distributed.nanny - WARNING - Restarting worker


CPU times: user 1min 40s, sys: 14.5 s, total: 1min 54s
Wall time: 10min 8s


KilledWorker: Attempted to run task 'finalize-hlgfinalizecompute-29b215d418c945d6a02ffcf82c013aae' on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://127.0.0.1:33923. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [14]:
xr.__version__

'2025.6.1'